# **Imports**

In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
import sys
print(sys.path)

['C:\\Program Files\\JetBrains\\PyCharm 2025.3.1.1\\plugins\\python-ce\\helpers\\jupyter_debug', 'C:\\Program Files\\JetBrains\\PyCharm 2025.3.1.1\\plugins\\python-ce\\helpers\\pydev', '..', 'C:\\Users\\Frank\\PycharmProjects\\BIOE 486 Applied Deep Learning\\Final_Project', 'C:\\Users\\Frank\\anaconda3\\envs\\bioe486\\python311.zip', 'C:\\Users\\Frank\\anaconda3\\envs\\bioe486\\DLLs', 'C:\\Users\\Frank\\anaconda3\\envs\\bioe486\\Lib', 'C:\\Users\\Frank\\anaconda3\\envs\\bioe486', '', 'C:\\Users\\Frank\\anaconda3\\envs\\bioe486\\Lib\\site-packages']


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from src.config import device, NUM_EPOCHS, NUM_CLASSES, LR_A, LR_B_HEAD, LR_B_BACKBONE
from src.crc_dataset import train_loader, val_loader, test_loader
from src.model import model_a, model_b
from src.train import run_experiment

# **Device Confirmation**

In [4]:
# Cell 2: Confirm environment
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"model_a backbone frozen: {not next(model_a.backbone.parameters()).requires_grad}")
print(f"model_b backbone frozen: {not next(model_b.backbone.parameters()).requires_grad}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4070
model_a backbone frozen: True
model_b backbone frozen: False


In [5]:
# Cell 1.5 — Smoke Test (run before full training)
print("=== Smoke Test ===")

# Grab one batch from train_loader
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
print(f"Batch shape: {images.shape}")   # Should be [32, 3, 224, 224]
print(f"Labels shape: {labels.shape}") # Should be [32]

# Forward pass through both models
with torch.no_grad():
    out_a = model_a(images)
    out_b = model_b(images)

print(f"Exp A output shape: {out_a.shape}")  # Should be [32, 9]
print(f"Exp B output shape: {out_b.shape}")  # Should be [32, 9]
print(f"Exp A output range: [{out_a.min():.2f}, {out_a.max():.2f}]")
print(f"Exp B output range: [{out_b.min():.2f}, {out_b.max():.2f}]")

# Confirm frozen/unfrozen
frozen = sum(1 for p in model_a.backbone.parameters() if not p.requires_grad)
total  = sum(1 for p in model_a.backbone.parameters())
print(f"\nExp A frozen params: {frozen}/{total}")
print(f"Exp B frozen params: 0/{total} (all unfrozen)")

print("\nSmoke test passed — safe to run full training.")

=== Smoke Test ===
Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Exp A output shape: torch.Size([32, 9])
Exp B output shape: torch.Size([32, 9])
Exp A output range: [-0.60, 0.77]
Exp B output range: [-0.72, 1.07]

Exp A frozen params: 159/159
Exp B frozen params: 0/159 (all unfrozen)

Smoke test passed — safe to run full training.


# **Define Optimizer A / Run Experiment A**

In [7]:
# Experiment A: frozen backbone, head only
optimizer_a = optim.Adam(model_a.parameters(), lr=LR_A)

print(f"--- Starting Experiment A (LR: {LR_A}) ---")

history_a, best_acc_a, elapsed_a, peak_mem_a = run_experiment(
    model=model_a,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_a,
    num_epochs=NUM_EPOCHS,
    experiment_name="Exp_A"
)

print(f"--- Experiment A Complete | Best Val Acc: {best_acc_a:.4f} | Time: {elapsed_a:.1f}s ---")

--- Starting Experiment A (LR: 0.001) ---
[Exp_A] Epoch 1/20 | Train loss: 0.0116 acc: 0.8903 | Val loss: 0.0053 acc: 0.9535
[Exp_A] Epoch 2/20 | Train loss: 0.0080 acc: 0.9173 | Val loss: 0.0046 acc: 0.9583
[Exp_A] Epoch 3/20 | Train loss: 0.0079 acc: 0.9163 | Val loss: 0.0043 acc: 0.9601
[Exp_A] Epoch 4/20 | Train loss: 0.0077 acc: 0.9179 | Val loss: 0.0040 acc: 0.9615
[Exp_A] Epoch 5/20 | Train loss: 0.0077 acc: 0.9184 | Val loss: 0.0042 acc: 0.9599
[Exp_A] Epoch 6/20 | Train loss: 0.0077 acc: 0.9184 | Val loss: 0.0038 acc: 0.9637
[Exp_A] Epoch 7/20 | Train loss: 0.0078 acc: 0.9174 | Val loss: 0.0039 acc: 0.9619
[Exp_A] Epoch 8/20 | Train loss: 0.0077 acc: 0.9186 | Val loss: 0.0038 acc: 0.9634
[Exp_A] Epoch 9/20 | Train loss: 0.0077 acc: 0.9193 | Val loss: 0.0037 acc: 0.9636
[Exp_A] Epoch 10/20 | Train loss: 0.0076 acc: 0.9208 | Val loss: 0.0038 acc: 0.9635
[Exp_A] Epoch 11/20 | Train loss: 0.0077 acc: 0.9190 | Val loss: 0.0037 acc: 0.9649
[Exp_A] Epoch 12/20 | Train loss: 0.0076 ac

# **Define Optimizer B / Run Experiment B**

In [8]:
# Experiment B: Full fine-tuning

# Setup the optimizer with the new parameters
optimizer_b = optim.Adam([
    {'params': model_b.backbone.parameters(), 'lr': LR_B_BACKBONE},
    {'params': model_b.head.parameters(),     'lr': LR_B_HEAD}
])

print(f"--- Starting Experiment B (backbone LR: {LR_B_BACKBONE}, head LR: {LR_B_HEAD}) ---")

scheduler_b = optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=NUM_EPOCHS)
history_b, best_acc_b, elapsed_b, peak_mem_b = run_experiment(
    model=model_b,
    scheduler = scheduler_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_b,
    num_epochs=NUM_EPOCHS,
    experiment_name="Exp_B"
)

print(f"--- Experiment B Complete | Best Val Acc: {best_acc_b:.4f} | Time: {elapsed_b:.1f}s ---")

--- Starting Experiment B (backbone LR: 1e-05, head LR: 0.0001) ---
[Exp_B] Epoch 1/20 | Train loss: 0.0105 acc: 0.9038 | Val loss: 0.0020 acc: 0.9809
[Exp_B] Epoch 2/20 | Train loss: 0.0029 acc: 0.9711 | Val loss: 0.0012 acc: 0.9883
[Exp_B] Epoch 3/20 | Train loss: 0.0020 acc: 0.9799 | Val loss: 0.0007 acc: 0.9930
[Exp_B] Epoch 4/20 | Train loss: 0.0014 acc: 0.9852 | Val loss: 0.0006 acc: 0.9944
[Exp_B] Epoch 5/20 | Train loss: 0.0011 acc: 0.9889 | Val loss: 0.0005 acc: 0.9949
[Exp_B] Epoch 6/20 | Train loss: 0.0009 acc: 0.9909 | Val loss: 0.0004 acc: 0.9963
[Exp_B] Epoch 7/20 | Train loss: 0.0007 acc: 0.9926 | Val loss: 0.0003 acc: 0.9963
[Exp_B] Epoch 8/20 | Train loss: 0.0006 acc: 0.9937 | Val loss: 0.0003 acc: 0.9961
[Exp_B] Epoch 9/20 | Train loss: 0.0005 acc: 0.9945 | Val loss: 0.0003 acc: 0.9970
[Exp_B] Epoch 10/20 | Train loss: 0.0005 acc: 0.9952 | Val loss: 0.0003 acc: 0.9967
[Exp_B] Epoch 11/20 | Train loss: 0.0004 acc: 0.9960 | Val loss: 0.0003 acc: 0.9970
[Exp_B] Epoch 12/

# **Comparison**

In [9]:
# Comparison Summary

print("Experiment A (Frozen Backbone)")
print(f"Accuracy:  {best_acc_a:.4f}")
print(f"Time:      {elapsed_a:.1f}s")
print(f"Peak VRAM: {peak_mem_a:.1f} MB")
print()

print("=" * 40)

print("Experiment B (Full Fine-Tune)")
print(f"Accuracy:  {best_acc_b:.4f}")
print(f"Time:      {elapsed_b:.1f}s")
print(f"Peak VRAM: {peak_mem_b:.1f} MB")
print()

# Trade-off Analysis
acc_diff = best_acc_b - best_acc_a
time_mult = elapsed_b / elapsed_a if elapsed_a > 0 else 0

print("Trade-off Summary")
print(f"Net Accuracy Change: {acc_diff:+.4f}")
print(f"Compute Cost Factor: {time_mult:.2f}x longer")

Experiment A (Frozen Backbone)
Accuracy:  0.9655
Time:      2488.0s
Peak VRAM: 554.4 MB

Experiment B (Full Fine-Tune)
Accuracy:  0.9983
Time:      5728.9s
Peak VRAM: 3069.3 MB

Trade-off Summary
Net Accuracy Change: +0.0328
Compute Cost Factor: 2.30x longer


In [2]:
print(type(history_a), type(history_b))
print("Exp A epochs:", len(history_a['train_loss']))
print("Exp B epochs:", len(history_b['train_loss']))

NameError: name 'history_a' is not defined